In [2]:
import torch
import cv2
from torchvision import transforms

In [ ]:
from ref import EViT

In [4]:
# Load trained model
model = torch.load("E:\model.pth", map_location="cpu")

OrderedDict([('encoder.patch_embeds.0.proj.weight',
              tensor([[[[-1.4780e-01, -1.2815e-01, -4.5077e-02,  ..., -7.6434e-02,
                         -1.8340e-02, -2.1574e-01],
                        [-3.0720e-01, -7.5050e-02,  4.1317e-01,  ...,  3.2869e-01,
                         -6.5461e-03, -3.2223e-01],
                        [-1.3749e-01,  2.6752e-01,  1.1700e+00,  ...,  8.3998e-01,
                          5.5697e-01, -1.7830e-02],
                        ...,
                        [ 6.7020e-02,  7.0810e-02,  4.8902e-01,  ...,  2.7234e-01,
                          7.6905e-02,  1.1849e-02],
                        [-9.7526e-02, -1.2737e-01, -3.1232e-01,  ..., -2.2012e-02,
                         -8.2651e-02, -5.2931e-02],
                        [ 2.3446e-01,  5.7507e-02, -9.8584e-02,  ..., -1.7032e-01,
                         -1.3689e-01, -6.9238e-02]],
              
                       [[ 1.7799e-01, -1.8312e-01, -3.7972e-01,  ..., -3.8581e-01,
          

In [ ]:
model.eval()

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
# Image transform
transform = transforms.Compose([
    transforms.ToTensor()
])

# Open webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # preprocess
    img = transform(frame).to(device)
    img = img.unsqueeze(0)

    with torch.no_grad():
        outputs = model(img)

    # Example for detection models (FasterRCNN style output)
    boxes = outputs[0]['boxes']
    scores = outputs[0]['scores']

    for box, score in zip(boxes, scores):
        if score > 0.5:
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)

    cv2.imshow("Real-time Detection", frame)

    if cv2.waitKey(1) == 27:
        break

cap.release()
cv2.destroyAllWindows()